# **Protein Secondary Structure Prediction using Graph Neural Network**

Proteins are chains of amino acids joined together by peptide bonds . Each amino acid in the chain is polar, i.e. it has separated positive and negative charged regions with a free carbonyl group, which can act as hydrogen bond acceptor and an NH group, which can act as hydrogen bond donor. These groups can therefore interact in the protein structure. The 20 amino acids can be classified according to the chemistry of the side chain which also plays an important structural role.

The protein structure can be considered as a sequence of secondary structure elements, such as α helices, β sheets and coils, which together constitute the overall three-dimensional configuration of the protein chain.

# **Import the Library**

In [2]:
import numpy as np                                     # linear algebra
import pandas as pd                                    # data processing, CSV file I/O (e.g. pd.read_csv)
import copy     
import torch#to copy list
from sklearn.model_selection import train_test_split   #to split dataset into train and test set
from sklearn.svm import SVC                            #to create svc instance
from sklearn.metrics import classification_report      #to create report for precision,recall,f1-score,accuracy
from sklearn import metrics                            #to get accuracy
from sklearn.model_selection import GridSearchCV
import torch
import torch.nn as nn
import torch.nn as nn
import torch_geometric.nn as pyg_nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv #to optimise the hyper-parameter
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report



In [3]:
torch.cuda.is_available()

True

In [4]:
qw= torch.cuda.current_device()     # The ID of the current GPU.
se = torch.cuda.get_device_name(id)  # The name of the specified GPU, where id is an integer.
tn = torch.cuda.device(id)           # The memory address of the specified GPU, where id is an integer.
dv = torch.cuda.device_count()  

print(tn,se +"\n", qw,dv)     # The amount of GPUs that are accessible.

<torch.cuda.device object at 0x000002529B778E90> NVIDIA GeForce RTX 4070 SUPER
 0 1


In [5]:
torch.zeros(1).cuda()

tensor([0.], device='cuda:0')

# Directory of Dataset
find the directory of dataset

# Load the dataset

**PDB Dataset**<br>
Load the .csv file into dataframe and see if it is load properly .

In [6]:
df = pd.read_csv(r"H:\\THESIS\\DATASET\\2022-08-03-ss.cleaned.csv")
df.head()

,pdb_id,chain_code,seq,sst8,sst3,len,has_nonstd_aa
0,1A30,C,EDL,CBC,CEC,3,False
1,1B05,B,KCK,CBC,CEC,3,False
2,1B0H,B,KAK,CBC,CEC,3,False
3,1B1H,B,KFK,CBC,CEC,3,False
4,1B2H,B,KAK,CBC,CEC,3,False


# **Data processing**

The PDB dataset is opened and processed using the python programming language. The 'seq' column has the primary sequence of protein and the 'sst8' has the seconday sequence of protein . The max length of any sequence set to 128 .
hasnonstdaa: whether the peptide contains nonstandard amino acids (B, O, U, X, or Z).
So those sequences are only taken which don't have nonstandard amino acids .

In [7]:
maxlen_seq = 128
input_seqs, target_seqs = df[['seq', 'sst8']][(df.len <= maxlen_seq) & (~df.has_nonstd_aa)].values.T
#input_grams = seq2ngrams(input_seqs)
print(input_seqs[0:5])

['EDL' 'KCK' 'KAK' 'KFK' 'KAK']


see the target sequence(secondary structure) and it's size .

In [8]:
print(target_seqs[0:5])
print(target_seqs.size)

['CBC' 'CBC' 'CBC' 'CBC' 'CBC']
103801


# **Looking for incomplete data**

Check the data whether there are differences in the number of characters of the primary structure and secondary structure (because the prediction of the secondary structure of the protein is included in the sequence labeling problem, it is certain that the number of characters of the primary structure and secondary structure is always the same)

In [9]:
for row in range(len(target_seqs)):
    secondary_lenth = len(target_seqs[row])
    primary_lenth = len(input_seqs[row])

    if(secondary_lenth != primary_lenth):
        print("(",row,") Secondary_Structure ->", target_seqs[row]," Primary_Structure -> ",input_seqs[row])

Find the total sequence of primary structure and secondary structure to find out if there is no anomaly .

In [10]:
secondary_count = 0
primary_count = 0
for row in range(len(target_seqs)):
    secondary_lenth = len(target_seqs[row])
    primary_lenth = len(input_seqs[row])
    secondary_count = secondary_count + secondary_lenth
    primary_count = primary_count + primary_lenth
    if(secondary_lenth != primary_lenth):
        print("(",row,") Secondary_Structure ->", target_seqs[row]," Primary_Structure -> ",input_seqs[row])

print("count of secondary structure : ",secondary_count)
print("count of primary structure : ",primary_count)

count of secondary structure :  8386644
count of primary structure :  8386644


# **Orthogonal Encoding - Target Labeling**

Every primary and secondary structure data is split so that it can be encoded into orthogonal form

**Split function**<br>
split the string sequence into character array .<br>
input -> string <br>
output -> array of character

In [11]:
def split(sequence):
    return [char for char in sequence]

input_seqs is 2D array which has string in each row. Make it character array in each row .<br>
primary structure from input_seqs go to primary_split .<br>
secondary structure from target_seqs go to secondary_split .

In [12]:
primary_split = []
secondary_split = []
for row in range(int(len(target_seqs)/40)):
    primary_split.append(split(input_seqs[row]))
    secondary_split.append(split(target_seqs[row]))


The results of the split primary and secondary structure of the protein are then converted into orthogonal encoding and target labeling. A switch case snippet for each amino acid in the primary structure of a protein as follows .<br>
Secondary structure character represent ->
1. H= α-helix
2. C= Loops and irregular elements
3. E= β-strand
4. B= β-bridge
5. G= 3-helix
6. I= π-helix
7. T= Turn
8. S= Bend

In [13]:
def orthogonal_primary(arg):
    switch = {
        'A' : np.array([1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]),  # 20 amino acids
        'C' : np.array([0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]),
        'E' : np.array([0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]),
        'D' : np.array([0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]),
        'G' : np.array([0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]),
        'F' : np.array([0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0]),
        'I' : np.array([0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0]),
        'H' : np.array([0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0]),
        'K' : np.array([0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0]),
        'M' : np.array([0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0]),
        'L' : np.array([0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0]),
        'N' : np.array([0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0]),
        'Q' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0]),
        'P' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0]),
        'S' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0]),
        'R' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0]),
        'T' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0]),
        'W' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0]),
        'V' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0]),
        'Y' : np.array([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1])
    }

    return switch.get(arg)

def orthogonal_secondary(arg):
    switch = {
        'H' : 0,                    # H= α-helix
        'C' : 1,                    # C= Loops and irregular elements
        'E' : 2,                    # E= β-strand
        'B' : 3,                    # B= β-bridge
        'G' : 4,                    # G= 3-helix
        'I' : 5,                    # I= π-helix
        'T' : 6,                    # T= Turn
        'S' : 7                     # S= Bend
    }

    return switch.get(arg)

For each character of primary structure and secondary structure use onehot key in the 20 amino acid .<br>
In secondary structure encode the 8 classification using 0-7 integer .

In [14]:
for row in range(len(primary_split)):
    sequence = primary_split[row]
    for col in range(len(sequence)):
        # print(sequence[col])
        sequence[col] = orthogonal_primary(sequence[col])
        #print(sequence[col])

In [15]:
for row in range(len(secondary_split)):
    sequenceS = secondary_split[row]
    for col in range(len(sequenceS)):
        sequenceS[col] = orthogonal_secondary(sequenceS[col])

In [16]:
primary_split[0:5]

[[array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0])],
 [array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])],
 [array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])],
 [array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])],
 [array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  

In [17]:
secondary_split[0:5]

[[1, 3, 1], [1, 3, 1], [1, 3, 1], [1, 3, 1], [1, 3, 1]]

# Make the Graph

**graph_sum2**<br>
this function take input 2 node (amino acid character's onehot key) and return the sum of 2 node .<br>
**graph_sum3**<br>
this function take input 3 node (amino acid character's onehot key) and return the sum of 3 node .

In [18]:
def graph_sum2(seq1,seq2):
    result = [None]*len(seq1)
    for col in range(len(seq1)):
        result[col] =  seq1[col]+seq2[col]
    return result


def graph_sum3(seq1,seq2,seq3):
    result = [None]*len(seq1)
    for col in range(len(seq1)):
        result[col] =  seq1[col]+seq2[col]+seq3[col]
    return result

**Graph of primary structure**<br>
The primary structure is a linear string of character/amino acid(node) .<br>
Example : ***ABBA***<br>
In Graph Neural Network , we take each node and sum the information value of it's adjacent node. As a result we find a new value for each node which is dependable for it's adjacent nodes .<br>

the border node will use graph_sum2 function as it has only 1 adjacent node and the rest will use graph_sum3 function .

In [19]:
graph_input = copy.deepcopy(primary_split)
for row in range(len(primary_split)):
    sequence = primary_split[row]
    graph_input[row][0]=graph_sum2(sequence[0],sequence[1])
    graph_input[row][len(sequence)-1]=graph_sum2(sequence[len(sequence)-1],sequence[len(sequence)-2])
    for col in range(1,len(sequence)-1):
        graph_input[row][col] = graph_sum3(sequence[col-1],sequence[col],sequence[col+1])

graph_input[0:5]

[[[np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0)],
  [np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0)],
  [np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0)]],
 [[np.int64(0),
   np.int64(1),
   n

Make the secondary structure in a array of data using targetY function .

In [20]:
def targetY(data_list):
    Y = []
    for i in range(len(data_list)):
        for j  in range(len(data_list[i])):
            Y.append(data_list[i][j])
    return Y

In [21]:
y_label = targetY(secondary_split)

In [22]:
print(len(y_label))
print(y_label[0:5])

18642
[1, 3, 1, 1, 3]


The data feature is formed using the window_padding_data function. This function will accept the size of the sliding window and sequence of the primary structure of the protein. In this function features will be processed such as adding padding 0 at the beginning and end and taking the features of the results of windowing so that the output data can be directly trained on the SVM model

In [23]:
def window_padding_data(size, sequence):
    num = int(size/2)
    #print("initial :",sequence[0])
    #print("")
    zeros = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
    for i in range(len(sequence)):
        for j in range(num):
            sequence[i].append(zeros)
            sequence[i].insert(0, zeros)
            #print(sequence[i])
            #print("")

    X = []
    temp = []

    for k in range(len(sequence)):
        #print(sequence[k])
        for l in range(len(sequence[k])-(size-1)):
            temp = sequence[k][l:l+size]
           # print(temp)
            X.append(temp)
            temp = []

    return X

In [24]:
X = window_padding_data(11,graph_input)
len(X)
X[0:5]

[[array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  [np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0)],
  [np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(1),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0),
   np.int64(0)],
  [

Before being entered into the Scikit-Learn SVM model, the data is reshape to follow the input size of the model. The data is reshaped to the length of X(primary structure) times multiplied by 600 (x window size and 20 orthogonal encoding sizes.)

In [25]:
np.set_printoptions(threshold=np.inf)
X = np.array(X)
y_label = np.array(y_label)
X = X.reshape(len(X),11*20)
print(X[0:5])

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0
  0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 

Data is split into training and testing data. Next will be calculated with SVM and see the Classification Report

In [26]:
#split the dataset into train set and test set
X_train, X_test, y_train, y_test = train_test_split(X, y_label, test_size = 0.20,random_state=54)

# Optimise the Hyper parameter

In [25]:
# # defining parameter range
# param_grid = {'C': [0.1, 1, 10, 100, 1000],
#              'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
#              'kernel': ['rbf']}

# grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3)

# # fitting the model for grid search
# grid.fit(X_train, y_train)

In [26]:
# print best parameter after tuning
#print(grid.best_params_)
#
# print how our model looks after hyper-parameter tuning
#print(grid.best_estimator_)

In [27]:
#grid_predictions = grid.predict(X_test)
#
# Define different kernel functions
#kernel_functions = ['linear', 'poly', 'rbf', 'sigmoid']
# print classification report
#print(classification_report(y_test, grid_predictions))

In [ ]:


# Defining parameter range
param_grid = {
    'C': [0.1, 1, 10, 100, 1000],
    'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf']
}

# Initializing GridSearchCV
grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=3)

# Fitting the model for grid search
grid.fit(X_train, y_train)

# Print the best parameters after tuning
print("Best parameters found: ", grid.best_params_)

# Making predictions
grid_predictions = grid.predict(X_test)

# Print classification report
print(classification_report(y_test, grid_predictions))

# Print the best estimator
print("Best estimator: ", grid.best_estimator_)


Fitting 5 folds for each of 25 candidates, totalling 125 fits


# Use the SVM to find the classification

In [27]:
#for i in range(1,101):
#    X_train, X_test, y_train, y_test = train_test_split(X, y_label, test_size = 0.20,random_state=i)
svc = SVC(kernel='rbf', gamma = 0.1, C=10)
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
y_true = y_test
#    print("i = ",i,"acc = ",metrics.accuracy_score(y_test, y_pred))
print("Accuracy = ",metrics.accuracy_score(y_test, y_pred)*100)
print(classification_report(y_true,y_pred))

Accuracy =  87.28881737731295
              precision    recall  f1-score   support

           0       0.95      0.70      0.81        60
           1       0.89      0.96      0.92      2597
           2       0.88      0.77      0.82       390
           3       0.83      0.68      0.75       155
           4       0.61      0.46      0.53        41
           6       0.73      0.53      0.61       174
           7       0.80      0.61      0.69       312

    accuracy                           0.87      3729
   macro avg       0.82      0.67      0.73      3729
weighted avg       0.87      0.87      0.87      3729



In [30]:
# EPSILON = 1e-10
# print('Mean Absolute Error(MAE):', metrics.mean_absolute_error(y_test, y_pred))
# # print('Mean Squared Error(MSE):', metrics.mean_squared_error(y_test, y_pred))
# # print('Root Mean Squared Error(RMSE):', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))
# print('Relative Absolute Error(RAE):', np.sum(np.abs(y_test - y_pred)) / (np.sum(np.abs(y_test - np.mean(y_test))) + EPSILON))
# print('Root Relative Squared Error(RRSE):', np.sqrt(np.sum(np.square(y_test - y_pred)) / np.sum(np.square(y_test - np.mean(y_test)))))

# **Cross validation**

In [31]:

def evaluate_model(cv):
    scores = cross_val_score(svc, X, y_label, scoring='accuracy', cv=cv, n_jobs=-1)
    return np.mean(scores), scores.min(), scores.max()


In [32]:
"""
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score

ideal, _, _ = evaluate_model(LeaveOneOut())
print('Ideal: %.3f' % ideal)
# define folds to test
folds = range(2,31)
# record mean and min/max of each set of results
means, mins, maxs = list(),list(),list()
"""

"\nfrom sklearn.model_selection import LeaveOneOut\nfrom sklearn.model_selection import cross_val_score\n\nideal, _, _ = evaluate_model(LeaveOneOut())\nprint('Ideal: %.3f' % ideal)\n# define folds to test\nfolds = range(2,31)\n# record mean and min/max of each set of results\nmeans, mins, maxs = list(),list(),list()\n"

In [33]:
'''
from sklearn.model_selection import cross_val_score
folds = range(2,31)
means, mins, maxs = list(),list(),list()
from sklearn.model_selection import KFold
for k in folds:
    cv = KFold(n_splits=k, shuffle=True, random_state=1)
    k_mean, k_min, k_max = evaluate_model(cv)
    print('> folds=%d, accuracy=%.3f (%.3f,%.3f)' % (k, k_mean, k_min, k_max))
    means.append(k_mean)
    mins.append(k_mean - k_min)
    maxs.append(k_max - k_mean)
'''

"\nfrom sklearn.model_selection import cross_val_score\nfolds = range(2,31)\nmeans, mins, maxs = list(),list(),list()\nfrom sklearn.model_selection import KFold\nfor k in folds:\n    cv = KFold(n_splits=k, shuffle=True, random_state=1)\n    k_mean, k_min, k_max = evaluate_model(cv)\n    print('> folds=%d, accuracy=%.3f (%.3f,%.3f)' % (k, k_mean, k_min, k_max))\n    means.append(k_mean)\n    mins.append(k_mean - k_min)\n    maxs.append(k_max - k_mean)\n"

In [34]:
'''
from matplotlib import pyplot
pyplot.errorbar(folds, means, yerr=[mins, maxs], fmt='o')
# plot the ideal case in a separate color
#pyplot.plot(folds, [ideal for _ in range(len(folds))], color='r')
# show the plot
pyplot.show()
'''

"\nfrom matplotlib import pyplot\npyplot.errorbar(folds, means, yerr=[mins, maxs], fmt='o')\n# plot the ideal case in a separate color\n#pyplot.plot(folds, [ideal for _ in range(len(folds))], color='r')\n# show the plot\npyplot.show()\n"

**Input and Output**
GNN

In [45]:
%pip install ipywidgets


   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 14.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [43]:
%pip install transformers

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.7 MB 7.5 MB/s eta 0:00:02
   ---------------------------------------  9.4/9.7 MB 29.4 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 27.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB ? eta -:--:--
   --------------------- ------------------ 1.3/2.4 MB 1.8 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 2.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


libs


In [2]:
%pip install --upgrade torch torchvision


  Using cached torch-2.5.1-cp311-cp311-win_amd64.whl.metadata (28 kB)
Using cached torch-2.5.1-cp311-cp311-win_amd64.whl (203.1 MB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 9.3 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.0.1+cu117
    Uninstalling torch-2.0.1+cu117:
      Successfully uninstalled torch-2.0.1+cu117
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.15.2+cu117
    Uninstalling torchvision-0.15.2+cu117:
      Successfully uninstalled torchvision-0.15.2+cu117
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.0.2+cu117 requires torch==2.0.1, but you have torch 2.5.1 which is incompatible.


In [2]:
#%pip install torchvision
%pip install transformers

  Using cached transformers-4.48.1-py3-none-any.whl.metadata (44 kB)
  Using cached huggingface_hub-0.27.1-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.21.0-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.5.2-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
Using cached transformers-4.48.1-py3-none-any.whl (9.7 MB)
Using cached huggingface_hub-0.27.1-py3-none-any.whl (450 kB)
Using cached safetensors-0.5.2-cp38-abi3-win_amd64.whl (303 kB)
Using cached tokenizers-0.21.0-cp39-abi3-win_amd64.whl (2.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:


import torchvision
torchvision.disable_beta_transforms_warning()
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
from tqdm import tqdm  # For progress bar

In [3]:
%pip install --upgrade numpy



Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: numpy in c:\users\kaush\.ai-navigator\micromamba\envs\cuda\lib\site-packages (2.0.2)
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/12.9 MB 11.2 MB/s eta 0:00:01
   ------------------------- -------------- 8.4/12.9 MB 24.8 MB/s eta 0:00:01
   ------------------------------- -------- 10.2/12.9 MB 18.2 MB/s eta 0:00:01
   ---------------------------------------  12.8/12.9 MB 17.1 MB/s eta 0:00:01
   ---------------------------------------- 12.9/12.9 MB 15.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.2 which is incompatible.


In [2]:
tensor = torch.tensor([1.0, 2.0, 3.0])
array = tensor.numpy()
print(array)

[1. 2. 3.]


In [11]:
structural_features = np.random.rand(len(df), 5)
print(structural_features)

[[0.55898583 0.08746519 0.42405965 0.59097338 0.07530318]
 [0.4623967  0.48774775 0.69785572 0.91191367 0.92967773]
 [0.95610666 0.06658737 0.28729141 0.51008196 0.53020945]
 ...
 [0.9755822  0.37468934 0.43780216 0.39923398 0.05324506]
 [0.65964286 0.82610921 0.17653929 0.17314006 0.07377683]
 [0.21603538 0.15930154 0.67326064 0.93808859 0.06826691]]


In [12]:


# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device available for running: ", device)



# Load pre-trained ProtBERT model and tokenizer
model_name = 'Rostlab/prot_bert_bfd'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name).to(device)
model.eval()  # Set model to evaluation mode

# Function to get embeddings from the transformer model
def get_sequence_embeddings(sequences, batch_size=128):
    embeddings = []
    for i in tqdm(range(0, len(sequences), batch_size), desc="Processing batches"):
        batch = sequences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {key: val.to(device) for key, val in inputs.items()}
        
        # Get embeddings from the model
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Average embeddings for each sequence in the batch
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)

# Apply the function to the dataset to get sequence embeddings
X_seq_embeddings = get_sequence_embeddings(df['seq'].tolist())


Device available for running:  cpu


: 

In [1]:
print(X_seq_embeddings.shape)

NameError: name 'X_seq_embeddings' is not defined

In [2]:
print(X_seq_embeddings[0])

NameError: name 'X_seq_embeddings' is not defined

In [54]:
# Combine the structural features and sequence embeddings by concatenation
X_combined = np.hstack([structural_features, X_seq_embeddings])
print(X_combined.shape)  # Check the shape of the combined features


(477153, 1029)


In [77]:
X_combined[0]

array([ 0.72124402,  0.89790874,  0.58174393, ..., -0.08683909,
       -0.1144791 , -0.01298949], shape=(1029,))

In [82]:

# Step 1: Define Node Features
# Assuming you have 3 nodes with 1029 features each
num_nodes = 3
node_features = torch.rand((num_nodes, 1029), dtype=torch.float)

# Step 2: Define Edges
# Ensure the edge indices are valid (between 0 and num_nodes - 1)
edges = torch.tensor(
    [
        [0, 1, 2],  # Source nodes
        [1, 2, 0]   # Target nodes
    ],
    dtype=torch.long
)

# Step 3: Create the Graph Data
graph_data = Data(x=node_features, edge_index=edges)

# Step 4: Validate Graph Data
print(f"Graph data:\n{graph_data}")
assert edges.max().item() < num_nodes, "Edge index contains invalid node indices!"

# Step 5: Debugging Outputs
print(f"Node features shape: {graph_data.x.shape}")
print(f"Edge index shape: {graph_data.edge_index.shape}")
print(f"Edge index:\n{graph_data.edge_index}")

Graph data:
Data(x=[3, 1029], edge_index=[2, 3])
Node features shape: torch.Size([3, 1029])
Edge index shape: torch.Size([2, 3])
Edge index:
tensor([[0, 1, 2],
        [1, 2, 0]])


In [78]:

from torch_geometric.data import Data

# Define the graph: Nodes and edges
# Here we assume each protein has nodes for each residue. For simplicity, we use an example graph.
num_nodes = len(df)  # Assume each protein is represented as one node in the graph
edges = torch.tensor([[0, 1, 2], [1, 2, 3]], dtype=torch.long).t().contiguous()  # Example edges (node 0 connected to node 1, etc.)

# Combine node features and edges for graph data
node_features = torch.tensor(X_combined[:2], dtype=torch.float)

# Create graph data object
graph_data = Data(x=node_features, edge_index=edges)


In [83]:


# Define a GCN model
class GCNModel(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNModel, self).__init__()
        self.conv1 = pyg_nn.GCNConv(in_channels, 128)
        self.conv2 = pyg_nn.GCNConv(128, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        return x

# Initialize the model
model = GCNModel(in_channels=X_combined.shape[1], out_channels=64)  # Example output channels


In [84]:
class TransformerLayer(nn.Module):
    def __init__(self, input_dim):
        super(TransformerLayer, self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim=input_dim, num_heads=8, dropout=0.1)

    def forward(self, x):
        x = x.unsqueeze(0)  # Add batch dimension for attention
        attn_output, _ = self.attn(x, x, x)
        return attn_output.squeeze(0)  # Remove the batch dimension

# Add transformer layer after GCN output
class GNNWithTransformer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GNNWithTransformer, self).__init__()
        self.gcn = GCNModel(in_channels, out_channels)
        self.transformer = TransformerLayer(out_channels)

    def forward(self, data):
        gcn_output = self.gcn(data)
        transformer_output = self.transformer(gcn_output)
        return transformer_output

# Initialize the model
gnn_transformer_model = GNNWithTransformer(in_channels=X_combined.shape[1], out_channels=64)


In [85]:
class FeatureFusionModel(nn.Module):
    def __init__(self, gcn_transformer_output_dim):
        super(FeatureFusionModel, self).__init__()
        self.fc1 = nn.Linear(gcn_transformer_output_dim * 2, 128)  # Concatenate GCN and transformer outputs
        self.fc2 = nn.Linear(128, 1)  # Final prediction layer

    def forward(self, gcn_output, transformer_output):
        fused_features = torch.cat([gcn_output, transformer_output], dim=1)
        x = torch.relu(self.fc1(fused_features))
        x = self.fc2(x)
        return x

# Initialize the fusion model
fusion_model = FeatureFusionModel(gcn_transformer_output_dim=64)


In [86]:
class FinalModel(nn.Module):
    def __init__(self, gcn_transformer_output_dim):
        super(FinalModel, self).__init__()
        self.gnn_transformer = GNNWithTransformer(
            in_channels=1029,  # Matches node feature dimension
            out_channels=gcn_transformer_output_dim
        )
        self.fusion = FeatureFusionModel(gcn_transformer_output_dim)

    def forward(self, data):
        gcn_output = self.gnn_transformer.gcn(data)
        transformer_output = self.gnn_transformer.transformer(gcn_output)
        return self.fusion(gcn_output, transformer_output)


In [87]:
print([graph_data][0])  


 

Data(x=[3, 1029], edge_index=[2, 3])


In [88]:
print(graph_data)  # Check graph structure
print(node_features.shape)  # Validate node features
print(output.shape)  # Ensure model outputs are correct


Data(x=[3, 1029], edge_index=[2, 3])
torch.Size([3, 1029])
torch.Size([477153, 1])


In [ ]:
# Ensure final_model is defined
final_model = FinalModel(gcn_transformer_output_dim=64).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001)

for epoch in range(100):
    for batch in X_combined:
        batch = batch.to(device) 
        
        # Debugging: Check if batch.y exists and is not None
        if not hasattr(batch, 'y') or batch.y is None:
            print("Warning: 'batch.y' is missing or None. Skipping this batch.")
            continue

        # Ensure batch.y is on the same device
        batch.y = batch.y.to(device)

        optimizer.zero_grad()
        output = final_model(batch)  # Forward pass
        
        # Ensure output and batch.y shapes match
        if output.size() != batch.y.size():
            print(f"Mismatch in shapes: output {output.size()}, batch.y {batch.y.size()}")
            continue
        
        loss = criterion(output, batch.y)  # Compute loss
        loss.backward()
        optimizer.step()

    # Optional: Print progress every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")